In [1]:
from scipy.stats import ttest_ind_from_stats
import numpy as np

def non_inferiority_ttest(mean1, stddev1, n1, mean2, stddev2, n2, relative_difference, equal_variance=False, increase_good=True):
    '''
    Perform a one-sided t-test with a non-inferiority threshold for two independent samples.
    mean1/2: group mean
    stddev1/2: standard deviation of each group
    n1/2: number of observations in each group
    relative_difference: threshold as a percentage of the base group (e.g. 0.1=10% difference)
    equal_variance: if False, uses Welch's t-test.
    increase_good: if True, Ho: mean2 <= mean1 - threshold. Else Ho: mean2 >= mean1 + threshold.
    Returns: 
    '''
    
    delta = relative_difference * mean1

    if increase_good:
        threshold = mean1 - delta
    else:
        threshold = mean1 + delta

    tstat, pval = ttest_ind_from_stats(mean1=threshold, 
                                       std1=stddev1, 
                                       nobs1=n1, 
                                       mean2=mean2, 
                                       std2=stddev2, 
                                       nobs2=n2, 
                                       equal_var=equal_variance)

    if increase_good:
        pvalue = pval/2.0
    else:
        pvalue = 1 - pval/2.0
    
    return tstat, pvalue

In [7]:
from statistics import mean, stdev


relative_difference_threshold = 0.1

DIALGOGS = 500


human_success_rate = 0.7386
human_success = [0] * (int((1-human_success_rate) * DIALGOGS) + 1) + [1] * int(human_success_rate * DIALGOGS)
print(len(human_success))
print(mean(human_success))

genV3_success_rate = 0.6944
genv3_success = [0] * (int((1-genV3_success_rate) * DIALGOGS) + 1) + [1] * int(genV3_success_rate * DIALGOGS)
print(len(genv3_success))
print(mean(genv3_success))


tstat, pval = non_inferiority_ttest(mean1=mean(human_success),
                                    stddev1=stdev(human_success), 
                                    n1=len(human_success), 
                                    mean2=mean(genv3_success), 
                                    stddev2=stdev(human_success), 
                                    n2=len(genv3_success), 
                                    relative_difference=0.125, 
                                    equal_variance=False, 
                                    increase_good=True)

print('One sided ttest: t value = {:.4f}, pval = {:.4f}'.format(tstat, pval))

500
0.738
500
0.694
One sided ttest: t value = -1.7332, pval = 0.0417
